In [7]:
import os
import random
import pandas as pd
import shutil
import git
import fnmatch
from tqdm import tqdm

# Change it to your google drive path where this notebook located.
drive_path = '/Users/samyiin/Projects/ZipfLawAnalysis'
os.chdir(drive_path)

from Utils.PythonParser import PythonFileCleaner
from Utils.NameParser import NameParser

In [8]:
df_users = pd.read_csv('Database/TempData/GatherData/Filter_4_SendUserEmail/Results/all_users.csv')

In [9]:
def cleanup_single_user(python_file_cleaner, dir_save_python_files_dir, dir_cleaned_python_files_dir):
    list_df_names = []
    # walk through the directory
    for foldername, subfolders, filenames in os.walk(dir_save_python_files_dir):
        for filename in filenames:
            if fnmatch.fnmatch(filename, '*.py'):
                python_file_path = os.path.join(foldername, filename)
                try:
                    python_file_cleaner.cleanup_python_file(python_file_path, dir_cleaned_python_files_dir)
                except:
                    # if we cannot cleanup this file then we just give up
                    pass
    # clear all the python 2to3 files we created
    python_file_cleaner.clear_2to3_created_files(dir_save_python_files_dir)
    
def cleanup_all_users(df_users):
    python_file_cleaner = PythonFileCleaner()
    for i in tqdm(range(len(df_users))):
        user_login = df_users.iloc[i].to_dict()['login']
        # go to the directory for this user
        user_directory_path = os.path.join('Database/UserData', str(user_login))
        dir_save_python_files_dir = os.path.join(user_directory_path, 'PythonFiles')
        dir_cleaned_python_files_dir = os.path.join(user_directory_path, 'CleanedPythonFiles')
        
        # "Cache" the results: see if this dir is marked as done
        finish_token_fp = os.path.join(user_directory_path, 'finish_OP2')
        # os.remove(finish_token_fp)
        if os.path.exists(finish_token_fp):
            continue

        # parse all this user's python files, we saved them under Database/UserData/<user_login>/CleanedPythonFiles/
        cleanup_single_user(python_file_cleaner, dir_save_python_files_dir, dir_cleaned_python_files_dir)
    
        # finish cleaning this user, mark a directory as done: write the finish token
        with open(finish_token_fp, "w") as token_file:
            pass  # Creating an empty file
        

cleanup_all_users(df_users)

100%|██████████████████████████████████████████████████████████████████████████████████████| 576/576 [00:00<00:00, 11040.82it/s]
